WISO100303 / Johannes Schmidt & Peter Regner

# **An introduction to scientific programming**

<br> <br> <br> <br><br> <br> <br> <br>

## Learning goals

By the end of this lecture, you should be able to:

- select, filter, and summarize data with pandas;
- work with a DataFrame index and datetime information;
- resample a time series to a different temporal resolution;
- group observations by calendar features such as weekday or month;
- inspect time-series data for missing or incomplete periods.

# Download data
We use a rather extensive data set during this class. The code below automatically downloads the data. The download needs to be done only once because Datalore stores the file with the notebook.

In [ ]:
# Download the data
def download_attached_files():
    import urllib.request
    import os.path
    import shutil
    fnames = {
              'entsoe-demand-shortened.pickle': 'https://files.boku.ac.at/filr/public-link/file-download/0d7483b89a02e28a019ada50b0960942/18940/5822641704919291825/entsoe-demand-shortened.pickle'
    }
    for fname, url in fnames.items():
        if not os.path.exists(fname):
            print(f'Downloading: {url}')
            urllib.request.urlretrieve(url, filename=fname)
            print(f'Download finished!')
        else:
            print("File already exists, not downloading again.")

download_attached_files()

In [ ]:
# It takes about 3min to download the ENTSO-E data from Google Colab. Use the
# code below to cache the data on your own Google Drive. You will need grant
# permissions to "Google Drive for Desktop" for some reason to make it work.
#
# from google.colab import drive
# import shutil
# drive.mount('/content/drive')
# if os.path.exists('/content/drive/MyDrive' + fname):
#    if not os.path.exists(fname):
#        shutil.copy('/content/drive/MyDrive' + fname, '/content/')
# else:
#    if not os.path.exists(fname):
#        urllib.request.urlretrieve(url, filename=fname)
#    shutil.copy('/content/' + fname, '/content/drive/MyDrive/')

# pandas

Today, we work with the `pandas` library. `pandas` is a library that allows us to handle mixed-type data frames. Unlike NumPy, which allows only one data type per array, each column in a pandas DataFrame can have a different data type. Think of `pandas` as providing Excel-like functionality in Python: you can sort data, aggregate data, perform calculations on data, etc. The advantage:
- You can automatize your tasks
- You can reproduce your analysis
- You can separate data and code

`pandas` is built on top of NumPy, so many operations you know from NumPy may work in `pandas` as well.

pandas DataFrames have column names, which can be used to access them (see below). They also have an index for rows, such as the date and time of a sample in the table. During class, we will learn to handle both column names and the index.

# Data example: which simple patterns can we identify in electricity demand?

Today we want to use the `pandas` library to study which temporal patterns we can find in Austrian electricity demand. When do people use more electricity? When do they use less electricity?

For that purpose, we use data provided by ENTSO-E, the "European association for the cooperation of transmission system operators". It provides hourly data for most European countries and informs us about the "load" on the network. We will look at the dataset, which spans from late 2014 to today, and aim to understand consumption patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# Load data

We load data from the file we just downloaded. It contains load measured on grids in the whole of Europe, provided by entso-e. You can download the raw CSV files from the entso-e website directly, but we have done so for you, as the download takes substantial time. The pickle file, which is hosted on a bokubox, contains all relevant data.

In [ ]:
power_demand = pd.read_pickle("entsoe-demand-shortened.pickle") 

**Note:** Pickle is [not a good file format for data exchange](https://nedbatchelder.com/blog/202006/pickles_nine_flaws.html). There are security dangers and possible compatibility issues. Unfortunately, all other formats (except pickle, NPZ, and CSV) require installing additional packages on Datalore. To avoid the hassle, we used pickle despite its risks.

What's in there?

In [ ]:
power_demand

That's a lot of information... let's get an overview! Which columns are there?

In [ ]:
power_demand.columns

In pandas, rows can also have names. They are called the index of the pandas DataFrame:

In [ ]:
power_demand.index

# Accessing data

The `dataframe.columns` attribute tells us which columns are available in our table. To access a column, you can use `dataframe_name["Column name"]`. 

In [ ]:
power_demand["AreaName"]

Which areas are there? The command `unique()` will give you a listing of unique entries in that column.

In [ ]:
power_demand["AreaName"].unique()

What do the abbreviations mean? Do you think Austria is there? But what if the list was very long and we do not want to check manually if we can find Austria in the list of unique countries?

We can use a comparison and `np.sum` to find out! Please observe that a Boolean value is treated as 1 if it is `True`, and 0 otherwise.

In [ ]:
np.sum(power_demand["AreaName"] == "AT CTY")

What does that answer tell us?

We can also access rows of the DataFrame by using the method `.loc`

In [ ]:
power_demand.loc['2015-01-01']

In [ ]:
power_demand.loc['2015-01-01 00:00:00']

In [ ]:
power_demand.loc['2015-01-01':'2015-02-02']

If we want to, we can also access a row or a column by its numerical index (similar to numpy). For that purpose, we have to use `.iloc`.

In [ ]:
power_demand.iloc[0, 0]

Let's get Austrian data! We can filter rows by the values of a column using `dataframe_name[dataframe_name["column_name"] == criterion]`. This is similar to how we filtered NumPy arrays.

In [ ]:
def filter_country(load, country):
    country_load = load[load["AreaName"] == country]
    return country_load


power_demand_at = filter_country(power_demand, "AT CTY")

In [ ]:
power_demand_at

## Exercise 1

Let's do some summary statistics. Calculate mean, standard deviation, min, max and the 25%, 50% and 75% quantile of the distribution of the Austrian data. Hint: There may be a single pandas function that does it for you...

In a second step, do the same for Germany. Does the * 10 rule hold? (everything in Germany is ten times as big as in Austria)

In [ ]:
# # # # # YOUR SOLUTION GOES HERE # # # # #

<div style="color:#555;border-top:1px solid #999;text-align:right;padding:4px;">End of exercise</div>

# Plotting pandas data + some additional index information

The nice thing about pandas is that one can simply use `.plot()` on the DataFrame, and pandas will create an appropriate plot of the underlying data. Now it is important to understand the index better because the x-axis will be the index. If no index is given, pandas automatically creates one.

In [ ]:
data = {'data1': [2, 1, 7],
        'data2': [3, 4, 5]}

# Creating a new DataFrame without an index. pandas will automatically create an index.
df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
df.plot()

In [ ]:
### Creating a new DataFrame with an integer index
df = pd.DataFrame(data, index=[1, 3, 7])

In [ ]:
df

In [ ]:
df.plot()

In [ ]:
# Creating a new DataFrame with a string index.
df = pd.DataFrame(data, index=["house", "car", "pet"])

In [ ]:
df

In [ ]:
df.plot()

Now let's plot the Austrian data. 

In [ ]:
def plot_data(data, xlabel="Time"):
    data["TotalLoadValue"].plot()
    plt.xlabel(xlabel)
    plt.ylabel("Electricity load (MW)")


plot_data(power_demand_at)

Hm... there may be outliers. But how complete is the Austrian data?

Observe: we use f-strings here. It is a very convenient way of merging strings with variables. More at the end of the lecture!

In [ ]:
def check_data_completeness(data):
    number_of_nas = np.sum(pd.isna(data["TotalLoadValue"]))
    number_of_0s = np.sum(data["TotalLoadValue"] == 0)

    print(f'The data contains {number_of_nas} NAs and {number_of_0s} zeros')


check_data_completeness(power_demand_at)

## Exercise 2

What is the time resolution of the dataset? There are several ways of finding it, try to find at least two.

In [ ]:
# # # # # YOUR SOLUTION GOES HERE # # # # #

<div style="color:#555;border-top:1px solid #999;text-align:right;padding:4px;">End of exercise</div>

# Data aggregation 

Let's aggregate the data to an hourly resolution. The `resample()` method allows us to do so (`1h` implies aggregation to a one-hour resolution; check the documentation to see which other resolutions are available). Please observe that you have to use a method that can actually aggregate data to a different temporal resolution, such as `mean`, `max`, `min`, or `sum`.

In [ ]:
power_demand_at_hourly = power_demand_at.resample('1h').mean(numeric_only=True)

In [ ]:
power_demand_at

There are non-numeric columns in the DataFrame! To fix the warning, we add `numeric_only=True` to the call to the `mean` function.

In [ ]:
power_demand_at_hourly = power_demand_at.resample('1h').mean(numeric_only=True)

In [ ]:
power_demand_at_hourly

In [ ]:

plot_data(power_demand_at_hourly)


Let's look into the data more closely and find some regularities...

In [ ]:
power_demand_at_yearly = power_demand_at_hourly.resample('YE').mean()

In [ ]:
power_demand_at_yearly.index

In [ ]:
plot_data(power_demand_at_yearly)

# Time indexing

Hm... 2014 and 2025 are incomplete. We should therefore select the correct period. This can be done by working with the time index of the DataFrame, using `dataframe_name.loc` plus a date range in brackets, i.e. `dataframe_name.loc['start-date':'end-date']`.

In [ ]:
power_demand_at_yearly.index

In [ ]:
power_demand_at_yearly_full_years = power_demand_at_yearly.loc['2015-01-01':'2024-12-31']

plot_data(power_demand_at_yearly_full_years)

In [ ]:
power_demand_at_yearly_full_years

Let's plot it relative to 2015.

In [ ]:
power_demand_at_yearly_2015 = power_demand_at_yearly_full_years.loc['2015-12-31 00:00:00']
power_demand_at_yearly_full_years_relative = 100 * power_demand_at_yearly_full_years / power_demand_at_yearly_2015
power_demand_at_yearly_full_years_relative.plot()

plt.ylabel('Relative to 2015 (%)');

In [ ]:
power_demand_at_yearly_full_years_relative

Hm... it is hard to tell whether there is a trend. The time series is too short, so let's neglect it for the moment. Is there monthly seasonality?

In [ ]:
power_demand_at_monthly = power_demand_at_hourly.resample('ME').mean()
power_demand_at_monthly = power_demand_at_monthly.loc['2015-01-01':'2024-12-31']

plot_data(power_demand_at_monthly)

hm... seems to be seasonal, right?

To understand it better, we could simply take the monthly average...

In [ ]:
power_demand_at_monthly_mean = power_demand_at.groupby(
    power_demand_at.index.month).mean(numeric_only=True)

plot_data(power_demand_at_monthly_mean, "Month of Year")

What happened here? First, `power_demand_at.index.month` simply returns the month of a time index.

In [ ]:
power_demand_at.index.month

With group_by we can calculate a function on subgroups of the original data. In our case, we calculate the `mean` for all values which belong to the same group, i.e. same month. Another example:

In [ ]:
data = {'data1': [2, 1, 7, 3, 3, 22, 2, 1, 44],
        'data2': [3, 4, 5, 3, 4, 4, 2, 2, 2]}

# Creates pandas DataFrame.
df = pd.DataFrame(data, index=[1, 1, 1, 1, 2, 3, 3, 3, 4])

In [ ]:
df

In [ ]:
df.groupby(df.index).sum()

In [ ]:
df.groupby(["data1"]).sum()

## Exercise 3

There are at least two other time scales over which data shows seasonality. Can you find them and plot them? Hint: power_demand_at_hourly.index.weekday may be very useful here.

In [ ]:
# # # # # YOUR SOLUTION GOES HERE # # # # #

<div style="color:#555;border-top:1px solid #999;text-align:right;padding:4px;">End of exercise</div>

# A small detour: f-strings

During class, we have used `f-strings`. These are useful for creating formatted strings that contain a mixture of expressions and strings. The syntax is `f''`. If one wants to output a variable, curly brackets are used: `{}`.

In [ ]:
x = 1
f'x is {x}'

In [ ]:
f'33*22={33*22}'

We can even do additional formatting such as rounding:

In [ ]:
import numpy as np

f'Pi rounded to two digits is {np.pi:.2f}'

In [ ]:
f'Pi rounded two 0 digits is {np.pi:.0f}'

**Caution!** This will work: 

In [ ]:
f'Power demand in Austria is {round(power_demand_at_yearly_full_years["TotalLoadValue"].loc["2015-12-31 00:00:00"])}'

This will also work:

In [ ]:
f"Power demand in Austria is {round(power_demand_at_yearly_full_years['TotalLoadValue'].loc['2015-12-31 00:00:00'])}"

But this won't: 

In [ ]:
f"Power demand in Austria is {round(power_demand_at_yearly_full_years["TotalLoadValue"].loc["2015-12-31 00:00:00"])}"